<a href="https://colab.research.google.com/github/hassanaldakn/hassanaldakn/blob/main/Scholar_PDF_Downloader_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests beautifulsoup4 pandas openpyxl ipywidgets

In [ ]:
import os
import re
import csv
import time
import json
import zipfile
from urllib.parse import urljoin, urlparse, unquote, quote

import requests
import pandas as pd
from bs4 import BeautifulSoup

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files


# =========================================================
# CONFIG
# =========================================================
DEFAULT_TIMEOUT = 30
USER_AGENT = "ScholarPDFDownloader/2.0 (Google Colab OA downloader)"
CROSSREF_BASE = "https://api.crossref.org"
UNPAYWALL_BASE = "https://api.unpaywall.org/v2"
OPENALEX_BASE = "https://api.openalex.org"
EUROPEPMC_BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest"

PDF_CONTENT_TYPES = {
    "application/pdf",
    "application/x-pdf",
    "application/acrobat",
    "applications/vnd.pdf",
    "text/pdf",
    "text/x-pdf",
}

APP_STATE = {
    "input_path": None,
    "results": [],
    "zip_path": None,
    "output_dir": None,
    "running": False,
}


# =========================================================
# BASIC HELPERS
# =========================================================
def session_with_headers():
    s = requests.Session()
    s.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "*/*",
    })
    return s


def ensure_folder(path):
    os.makedirs(path, exist_ok=True)


def clean_filename(name: str, max_len: int = 180) -> str:
    name = re.sub(r'[\\/:*?"<>|]+', "_", str(name))
    name = re.sub(r"\s+", " ", name).strip()
    return name[:max_len] if len(name) > max_len else name


def slugify_author(author: str) -> str:
    author = str(author or "").strip()
    if not author:
        return "UnknownAuthor"
    author = re.sub(r"[^A-Za-z0-9_\- ]+", "", author)
    author = re.sub(r"\s+", "_", author).strip("_")
    return author or "UnknownAuthor"


def safe_year(text):
    if text is None:
        return "UnknownYear"
    m = re.search(r"(19|20)\d{2}", str(text))
    return m.group(0) if m else "UnknownYear"


def is_doi(text: str) -> bool:
    text = str(text).strip()
    return bool(re.match(r"^(https?://(dx\.)?doi\.org/)?10\.\S+/\S+$", text, re.I))


def normalize_doi(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"^https?://(dx\.)?doi\.org/", "", text, flags=re.I)
    return text.strip()


def is_url(text: str) -> bool:
    try:
        p = urlparse(str(text).strip())
        return p.scheme in ("http", "https") and bool(p.netloc)
    except Exception:
        return False


def is_scopus_url(text: str) -> bool:
    text = str(text).lower()
    return (
        "scopus.com/inward/record.uri" in text
        or "scopus.com/record/display.uri" in text
    )


def parse_year_from_crossref(meta: dict):
    for key in ["published-print", "published-online", "published", "issued", "created"]:
        parts = meta.get(key, {}).get("date-parts", [])
        if parts and parts[0]:
            return str(parts[0][0])
    return ""


def first_author_from_crossref(meta: dict):
    authors = meta.get("author") or []
    if authors:
        family = authors[0].get("family") or ""
        given = authors[0].get("given") or ""
        return family or given or "UnknownAuthor"
    return "UnknownAuthor"


def title_from_crossref(meta: dict):
    titles = meta.get("title") or []
    return titles[0] if titles else ""


def extract_doi_from_text(text: str):
    if not text:
        return ""
    m = re.search(r"\b(10\.\S+/\S+)\b", str(text), flags=re.I)
    if not m:
        return ""
    doi = m.group(1).rstrip(").,;]")
    return doi


def normalize_possible_doi(text: str):
    if not text:
        return ""
    text = str(text).strip()
    if is_doi(text):
        return normalize_doi(text)
    doi = extract_doi_from_text(text)
    return doi


# =========================================================
# INPUT LOADING
# =========================================================
def lower_map(columns):
    return {str(c).strip().lower(): c for c in columns}


def find_col(df, candidates):
    cmap = lower_map(df.columns)
    for name in candidates:
        if name in cmap:
            return cmap[name]
    return None


def choose_columns(df):
    doi_col = find_col(df, [
        "doi", "document doi", "article doi", "source doi", "identifier doi"
    ])
    url_col = find_col(df, [
        "url", "link", "article url", "source url", "document url", "full text url"
    ])
    citation_col = find_col(df, [
        "full citation", "citation", "reference", "references", "ref", "bibliography"
    ])
    title_col = find_col(df, [
        "title", "article title", "document title"
    ])
    authors_col = find_col(df, [
        "authors", "author", "first author"
    ])
    year_col = find_col(df, [
        "year", "publication year", "published", "publication date", "date"
    ])
    return doi_col, url_col, citation_col, title_col, authors_col, year_col


def load_inputs_from_file(path: str):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".txt":
        items = []
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f, start=1):
                line = line.strip()
                if line and not line.startswith("#"):
                    items.append({
                        "rownum": i,
                        "raw": line,
                        "doi": normalize_possible_doi(line),
                        "url": line if is_url(line) else "",
                        "citation": line,
                        "title": "",
                        "authors": "",
                        "year": "",
                    })
        return items, "TXT lines"

    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file. Use TXT, CSV, XLSX, or XLS.")

    if df.empty:
        return [], "empty file"

    doi_col, url_col, citation_col, title_col, authors_col, year_col = choose_columns(df)

    items = []
    for idx, row in df.iterrows():
        doi_val = str(row[doi_col]).strip() if doi_col and pd.notna(row[doi_col]) else ""
        url_val = str(row[url_col]).strip() if url_col and pd.notna(row[url_col]) else ""
        cit_val = str(row[citation_col]).strip() if citation_col and pd.notna(row[citation_col]) else ""
        tit_val = str(row[title_col]).strip() if title_col and pd.notna(row[title_col]) else ""
        aut_val = str(row[authors_col]).strip() if authors_col and pd.notna(row[authors_col]) else ""
        yr_val = str(row[year_col]).strip() if year_col and pd.notna(row[year_col]) else ""

        if not any([doi_val, url_val, cit_val, tit_val]):
            continue

        items.append({
            "rownum": idx + 2,  # Excel-like row number including header row
            "raw": cit_val or doi_val or url_val or tit_val,
            "doi": normalize_possible_doi(doi_val or cit_val),
            "url": url_val if is_url(url_val) else "",
            "citation": cit_val,
            "title": tit_val,
            "authors": aut_val,
            "year": yr_val,
        })

    used = {
        "doi_col": str(doi_col) if doi_col else "",
        "url_col": str(url_col) if url_col else "",
        "citation_col": str(citation_col) if citation_col else "",
        "title_col": str(title_col) if title_col else "",
        "authors_col": str(authors_col) if authors_col else "",
        "year_col": str(year_col) if year_col else "",
    }
    return items, used


# =========================================================
# API LOOKUPS
# =========================================================
def crossref_get_work(session, doi):
    url = f"{CROSSREF_BASE}/works/{quote(doi, safe='')}"
    r = session.get(url, timeout=DEFAULT_TIMEOUT)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json().get("message", {})


def crossref_search_bibliographic(session, query, rows=5):
    params = {
        "query.bibliographic": query,
        "rows": rows,
        "select": "DOI,title,author,issued,published-print,published-online,container-title,score",
    }
    r = session.get(f"{CROSSREF_BASE}/works", params=params, timeout=DEFAULT_TIMEOUT)
    r.raise_for_status()
    return r.json().get("message", {}).get("items", [])


def openalex_search(session, query, per_page=5):
    params = {
        "search": query,
        "per-page": per_page,
        "mailto": EMAIL_VALUE.strip() if EMAIL_VALUE.strip() else None,
    }
    params = {k: v for k, v in params.items() if v}
    r = session.get(f"{OPENALEX_BASE}/works", params=params, timeout=DEFAULT_TIMEOUT)
    r.raise_for_status()
    return r.json().get("results", [])


def openalex_get_by_doi(session, doi):
    doi_url = f"https://doi.org/{doi}"
    encoded = quote(doi_url, safe="")
    url = f"{OPENALEX_BASE}/works/https://doi.org/{encoded}"
    r = session.get(url, timeout=DEFAULT_TIMEOUT)
    if r.status_code == 404:
        return None
    if not r.ok:
        return None
    return r.json()


def europepmc_search(session, query, page_size=5):
    params = {
        "query": query,
        "format": "json",
        "pageSize": page_size,
    }
    r = session.get(f"{EUROPEPMC_BASE}/search", params=params, timeout=DEFAULT_TIMEOUT)
    r.raise_for_status()
    return r.json().get("resultList", {}).get("result", [])


def unpaywall_lookup(session, doi, email):
    url = f"{UNPAYWALL_BASE}/{quote(doi, safe='')}"
    r = session.get(url, params={"email": email}, timeout=DEFAULT_TIMEOUT)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def choose_best_unpaywall_pdf(upw):
    if not upw:
        return None, None

    best = upw.get("best_oa_location") or {}
    if best.get("url_for_pdf"):
        return best.get("url_for_pdf"), best.get("url")

    for loc in upw.get("oa_locations", []):
        if loc.get("url_for_pdf"):
            return loc["url_for_pdf"], loc.get("url")

    if best.get("url"):
        return None, best.get("url")

    return None, None


def resolve_doi_landing_page(session, doi):
    url = f"https://doi.org/{doi}"
    r = session.get(url, allow_redirects=True, timeout=DEFAULT_TIMEOUT)
    if r.ok:
        return r.url
    return None


# =========================================================
# HTML / PDF HELPERS
# =========================================================
def is_probably_pdf_response(resp: requests.Response) -> bool:
    ctype = (resp.headers.get("Content-Type") or "").split(";")[0].strip().lower()
    if ctype in PDF_CONTENT_TYPES:
        return True

    cd = (resp.headers.get("Content-Disposition") or "").lower()
    if ".pdf" in cd:
        return True

    if resp.url.lower().endswith(".pdf"):
        return True

    return False


def extract_pdf_links_from_html(html: str, page_url: str):
    soup = BeautifulSoup(html, "html.parser")
    candidates = []

    for tag in soup.find_all("meta"):
        content = tag.get("content")
        if not content:
            continue
        key = (tag.get("name", "") + " " + tag.get("property", "")).lower()
        if ".pdf" in content.lower() or "pdf" in key:
            candidates.append(urljoin(page_url, content))

    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(" ", strip=True).lower()
        full = urljoin(page_url, href)
        if ".pdf" in href.lower():
            candidates.append(full)
        elif "pdf" in text:
            candidates.append(full)
        elif re.search(r"/pdf($|[/?])", href, re.I):
            candidates.append(full)

    seen = set()
    uniq = []
    for x in candidates:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq


def try_page_for_pdf(session, page_url):
    r = session.get(page_url, allow_redirects=True, timeout=DEFAULT_TIMEOUT)
    if not r.ok:
        return False, None, f"Landing page HTTP {r.status_code}"

    if is_probably_pdf_response(r):
        return True, r.url, "Direct PDF from landing page"

    links = extract_pdf_links_from_html(r.text, r.url)
    for link in links:
        rr = session.get(link, stream=True, allow_redirects=True, timeout=DEFAULT_TIMEOUT)
        if rr.ok and is_probably_pdf_response(rr):
            rr.close()
            return True, link, "PDF link found in HTML"

    return False, None, "No downloadable PDF found on landing page"


def save_pdf_from_url(session, pdf_url, out_dir, final_name):
    r = session.get(pdf_url, stream=True, allow_redirects=True, timeout=DEFAULT_TIMEOUT)
    if not r.ok:
        return False, f"HTTP {r.status_code}"

    if not is_probably_pdf_response(r):
        return False, f"Not a PDF (Content-Type={r.headers.get('Content-Type')})"

    path = os.path.join(out_dir, clean_filename(final_name))
    if not path.lower().endswith(".pdf"):
        path += ".pdf"

    with open(path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 64):
            if chunk:
                f.write(chunk)

    return True, path


# =========================================================
# METADATA RESOLUTION
# =========================================================
def build_filename(rownum, author, year):
    return f"{rownum}_{slugify_author(author)}_{safe_year(year)}.pdf"


def metadata_from_openalex(work):
    author = "UnknownAuthor"
    year = ""
    title = work.get("display_name", "") or ""
    doi = ""

    ids = work.get("ids") or {}
    doi_url = ids.get("doi") or ""
    if doi_url.lower().startswith("https://doi.org/"):
        doi = doi_url.replace("https://doi.org/", "").strip()

    authorships = work.get("authorships") or []
    if authorships:
        author = (authorships[0].get("author") or {}).get("display_name") or "UnknownAuthor"

    year = str(work.get("publication_year") or "")
    pdf_url = None

    oa = work.get("open_access") or {}
    if oa.get("oa_url"):
        pdf_url = oa.get("oa_url")

    return {
        "doi": doi,
        "title": title,
        "author": author,
        "year": year,
        "pdf_url": pdf_url,
        "landing_url": work.get("primary_location", {}).get("landing_page_url", ""),
        "source": "OpenAlex",
    }


def metadata_from_europepmc(rec):
    author = rec.get("authorString", "") or "UnknownAuthor"
    year = rec.get("pubYear", "") or ""
    title = rec.get("title", "") or ""
    doi = rec.get("doi", "") or ""
    pdf_url = ""
    landing_url = ""

    if rec.get("isOpenAccess") == "Y":
        if rec.get("fullTextUrlList", {}).get("fullTextUrl"):
            urls = rec["fullTextUrlList"]["fullTextUrl"]
            if isinstance(urls, list):
                for u in urls:
                    if u.get("documentStyle", "").lower() == "pdf":
                        pdf_url = u.get("url", "")
                        break
                    landing_url = landing_url or u.get("url", "")
            elif isinstance(urls, dict):
                if urls.get("documentStyle", "").lower() == "pdf":
                    pdf_url = urls.get("url", "")
                landing_url = urls.get("url", "")

    first_author = author.split(",")[0].strip() if author else "UnknownAuthor"

    return {
        "doi": doi,
        "title": title,
        "author": first_author,
        "year": year,
        "pdf_url": pdf_url,
        "landing_url": landing_url,
        "source": "EuropePMC",
    }


def resolve_record_metadata(session, row, email):
    """
    Return best metadata:
    {
      doi, title, author, year, pdf_url, landing_url, source, note
    }
    """
    raw_doi = normalize_possible_doi(row.get("doi", ""))
    raw_url = str(row.get("url", "")).strip()
    citation = str(row.get("citation", "")).strip()
    title = str(row.get("title", "")).strip()
    row_author = str(row.get("authors", "")).strip()
    row_year = str(row.get("year", "")).strip()

    base = {
        "doi": raw_doi,
        "title": title,
        "author": row_author or "UnknownAuthor",
        "year": row_year,
        "pdf_url": "",
        "landing_url": raw_url,
        "source": "",
        "note": "",
    }

    # 1) DOI direct
    if raw_doi:
        try:
            meta = crossref_get_work(session, raw_doi)
            if meta:
                base.update({
                    "doi": raw_doi,
                    "title": title_from_crossref(meta) or base["title"],
                    "author": first_author_from_crossref(meta) or base["author"],
                    "year": parse_year_from_crossref(meta) or base["year"],
                    "source": "Crossref DOI",
                })
        except Exception:
            pass

        try:
            upw = unpaywall_lookup(session, raw_doi, email)
            pdf_url, landing = choose_best_unpaywall_pdf(upw)
            if pdf_url:
                base["pdf_url"] = pdf_url
            if landing and not base["landing_url"]:
                base["landing_url"] = landing
            if upw:
                base["source"] = (base["source"] + " + Unpaywall").strip(" +")
        except Exception:
            pass

        if not base["pdf_url"]:
            try:
                work = openalex_get_by_doi(session, raw_doi)
                if work:
                    oa = metadata_from_openalex(work)
                    base["title"] = oa["title"] or base["title"]
                    base["author"] = oa["author"] or base["author"]
                    base["year"] = oa["year"] or base["year"]
                    base["pdf_url"] = oa["pdf_url"] or base["pdf_url"]
                    base["landing_url"] = oa["landing_url"] or base["landing_url"]
                    base["source"] = (base["source"] + " + OpenAlex").strip(" +")
            except Exception:
                pass

        if not base["landing_url"]:
            try:
                landing = resolve_doi_landing_page(session, raw_doi)
                if landing:
                    base["landing_url"] = landing
            except Exception:
                pass

        return base

    # 2) URL direct
    if raw_url and not is_scopus_url(raw_url):
        base["source"] = "URL input"
        return base

    # 3) Citation or title via Crossref bibliographic search
    query = citation or title
    if query:
        try:
            items = crossref_search_bibliographic(session, query, rows=5)
            if items:
                best = items[0]
                doi = best.get("DOI", "") or ""
                base.update({
                    "doi": doi,
                    "title": title_from_crossref(best) or base["title"],
                    "author": first_author_from_crossref(best) or base["author"],
                    "year": parse_year_from_crossref(best) or base["year"],
                    "source": "Crossref bibliographic",
                })

                if doi:
                    try:
                        upw = unpaywall_lookup(session, doi, email)
                        pdf_url, landing = choose_best_unpaywall_pdf(upw)
                        if pdf_url:
                            base["pdf_url"] = pdf_url
                        if landing:
                            base["landing_url"] = landing
                        base["source"] += " + Unpaywall"
                    except Exception:
                        pass

                    if not base["pdf_url"]:
                        try:
                            work = openalex_get_by_doi(session, doi)
                            if work:
                                oa = metadata_from_openalex(work)
                                base["pdf_url"] = oa["pdf_url"] or base["pdf_url"]
                                base["landing_url"] = oa["landing_url"] or base["landing_url"]
                                base["source"] += " + OpenAlex"
                        except Exception:
                            pass

                return base
        except Exception:
            pass

    # 4) OpenAlex search by title/citation
    if query:
        try:
            results = openalex_search(session, query, per_page=5)
            if results:
                oa = metadata_from_openalex(results[0])
                base.update({
                    "doi": oa["doi"] or base["doi"],
                    "title": oa["title"] or base["title"],
                    "author": oa["author"] or base["author"],
                    "year": oa["year"] or base["year"],
                    "pdf_url": oa["pdf_url"] or base["pdf_url"],
                    "landing_url": oa["landing_url"] or base["landing_url"],
                    "source": "OpenAlex search",
                })

                if base["doi"] and not base["pdf_url"]:
                    try:
                        upw = unpaywall_lookup(session, base["doi"], email)
                        pdf_url, landing = choose_best_unpaywall_pdf(upw)
                        if pdf_url:
                            base["pdf_url"] = pdf_url
                        if landing:
                            base["landing_url"] = landing or base["landing_url"]
                        base["source"] += " + Unpaywall"
                    except Exception:
                        pass

                return base
        except Exception:
            pass

    # 5) Europe PMC search
    if query:
        try:
            recs = europepmc_search(session, query, page_size=5)
            if recs:
                ep = metadata_from_europepmc(recs[0])
                base.update({
                    "doi": ep["doi"] or base["doi"],
                    "title": ep["title"] or base["title"],
                    "author": ep["author"] or base["author"],
                    "year": ep["year"] or base["year"],
                    "pdf_url": ep["pdf_url"] or base["pdf_url"],
                    "landing_url": ep["landing_url"] or base["landing_url"],
                    "source": "EuropePMC search",
                })
                return base
        except Exception:
            pass

    if raw_url and is_scopus_url(raw_url):
        base["note"] = "Scopus record URL detected; cannot use it as a reliable PDF source."
        base["source"] = "Scopus URL only"
        return base

    base["note"] = "Could not resolve metadata from DOI/URL/citation/title."
    return base


# =========================================================
# DOWNLOAD LOGIC
# =========================================================
def process_row(session, row, out_dir, email):
    rownum = row.get("rownum", "")
    result = {
        "rownum": rownum,
        "input_raw": row.get("raw", ""),
        "status": "",
        "doi": "",
        "title": "",
        "first_author": "",
        "year": "",
        "source": "",
        "input_url": row.get("url", ""),
        "resolved_landing_url": "",
        "pdf_url": "",
        "saved_file": "",
        "message": "",
    }

    try:
        meta = resolve_record_metadata(session, row, email)

        doi = meta.get("doi", "") or ""
        title = meta.get("title", "") or ""
        author = meta.get("author", "") or row.get("authors", "") or "UnknownAuthor"
        year = meta.get("year", "") or row.get("year", "") or "UnknownYear"
        pdf_url = meta.get("pdf_url", "") or ""
        landing_url = meta.get("landing_url", "") or row.get("url", "") or ""
        source = meta.get("source", "") or ""
        note = meta.get("note", "") or ""

        result["doi"] = doi
        result["title"] = title
        result["first_author"] = author
        result["year"] = year
        result["source"] = source
        result["resolved_landing_url"] = landing_url
        result["pdf_url"] = pdf_url

        final_name = build_filename(rownum, author, year)

        # A) direct known PDF URL
        if pdf_url:
            ok, msg = save_pdf_from_url(session, pdf_url, out_dir, final_name)
            if ok:
                result["status"] = "downloaded"
                result["saved_file"] = msg
                result["message"] = f"Downloaded from resolved PDF URL ({source})"
                return result

        # B) raw URL may itself be PDF or page
        raw_url = row.get("url", "")
        if raw_url and not is_scopus_url(raw_url):
            ok, direct_pdf_or_none, why = try_page_for_pdf(session, raw_url)
            if ok and direct_pdf_or_none:
                ok2, msg2 = save_pdf_from_url(session, direct_pdf_or_none, out_dir, final_name)
                if ok2:
                    result["status"] = "downloaded"
                    result["saved_file"] = msg2
                    result["pdf_url"] = direct_pdf_or_none
                    result["message"] = f"Downloaded from input URL page ({why})"
                    return result

        # C) resolved landing page
        if landing_url and not is_scopus_url(landing_url):
            ok, direct_pdf_or_none, why = try_page_for_pdf(session, landing_url)
            if ok and direct_pdf_or_none:
                ok2, msg2 = save_pdf_from_url(session, direct_pdf_or_none, out_dir, final_name)
                if ok2:
                    result["status"] = "downloaded"
                    result["saved_file"] = msg2
                    result["pdf_url"] = direct_pdf_or_none
                    result["message"] = f"Downloaded from landing page ({why})"
                    return result

        result["status"] = "skipped"
        result["message"] = note or "No open-access PDF found."
        return result

    except requests.RequestException as e:
        result["status"] = "error"
        result["message"] = f"Network error: {e}"
        return result
    except Exception as e:
        result["status"] = "error"
        result["message"] = f"Unexpected error: {e}"
        return result


def write_report(rows, report_path):
    df = pd.DataFrame(rows)
    df.to_csv(report_path, index=False, encoding="utf-8")


def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files_in_root in os.walk(folder_path):
            for file_name in files_in_root:
                abs_path = os.path.join(root, file_name)
                rel_path = os.path.relpath(abs_path, folder_path)
                zf.write(abs_path, arcname=rel_path)


# =========================================================
# UI
# =========================================================
EMAIL_VALUE = ""

title_html = widgets.HTML(
    value="<h2>Scholar PDF Downloader (Colab)</h2><p>Input can contain DOI, URL, full citation, or title.</p>"
)

email_widget = widgets.Text(
    value="",
    placeholder="your_email@example.com",
    description="Email:",
    layout=widgets.Layout(width="420px")
)

output_widget = widgets.Text(
    value="downloads",
    description="Folder:",
    layout=widgets.Layout(width="280px")
)

delay_widget = widgets.FloatText(
    value=0.3,
    description="Delay:",
    layout=widgets.Layout(width="180px")
)

upload_btn = widgets.Button(description="Upload file", button_style="info", icon="upload")
run_btn = widgets.Button(description="Run", button_style="success", icon="play")
zip_btn = widgets.Button(description="Download ZIP", button_style="warning", icon="download", disabled=True)
clear_btn = widgets.Button(description="Clear", icon="trash")

file_html = widgets.HTML(value="<b>Input file:</b> None")
cols_html = widgets.HTML(value="<b>Detected columns:</b> None")
status_html = widgets.HTML(value="<b>Status:</b> Ready")

progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Progress:",
    layout=widgets.Layout(width="700px")
)

log_output = widgets.Output(layout={"border": "1px solid #ccc", "height": "220px", "overflow_y": "auto"})
table_output = widgets.Output()


def log(msg):
    with log_output:
        print(msg)


def show_results():
    with table_output:
        clear_output(wait=True)
        if not APP_STATE["results"]:
            print("No results yet.")
            return
        display(pd.DataFrame(APP_STATE["results"]))


def reset_outputs():
    APP_STATE["results"] = []
    APP_STATE["zip_path"] = None
    progress.value = 0
    progress.max = 100
    progress.bar_style = ""
    zip_btn.disabled = True
    status_html.value = "<b>Status:</b> Ready"
    with log_output:
        clear_output(wait=True)
    with table_output:
        clear_output(wait=True)


def on_upload_clicked(b):
    if APP_STATE["running"]:
        return

    reset_outputs()
    uploaded = files.upload()
    if not uploaded:
        status_html.value = "<b>Status:</b> No file uploaded"
        return

    input_name = list(uploaded.keys())[0]
    APP_STATE["input_path"] = input_name

    try:
        items, info = load_inputs_from_file(input_name)
        file_html.value = f"<b>Input file:</b> {input_name}"
        cols_html.value = f"<b>Detected columns:</b> {json.dumps(info, ensure_ascii=False)}"
        status_html.value = f"<b>Status:</b> File loaded ({len(items)} rows)"
        log(f"Loaded file: {input_name}")
        log(f"Detected: {info}")
        log(f"Rows found: {len(items)}")
    except Exception as e:
        status_html.value = "<b>Status:</b> Failed to read file"
        log(f"Error: {e}")


def on_run_clicked(b):
    global EMAIL_VALUE

    if APP_STATE["running"]:
        return

    reset_outputs()

    input_path = APP_STATE.get("input_path")
    EMAIL_VALUE = email_widget.value.strip()
    out_dir = output_widget.value.strip() or "downloads"
    delay = max(0.0, float(delay_widget.value))

    if not input_path or not os.path.exists(input_path):
        status_html.value = "<b>Status:</b> Upload a file first"
        log("No input file.")
        return

    if not EMAIL_VALUE:
        status_html.value = "<b>Status:</b> Email is required"
        log("Unpaywall email is required.")
        return

    try:
        items, info = load_inputs_from_file(input_path)
    except Exception as e:
        status_html.value = "<b>Status:</b> Failed to parse file"
        log(f"Error parsing file: {e}")
        return

    if not items:
        status_html.value = "<b>Status:</b> No usable rows found"
        log("No usable rows found.")
        return

    ensure_folder(out_dir)
    APP_STATE["output_dir"] = out_dir
    APP_STATE["running"] = True

    file_html.value = f"<b>Input file:</b> {input_path}"
    cols_html.value = f"<b>Detected columns:</b> {json.dumps(info, ensure_ascii=False)}"
    progress.max = len(items)
    progress.value = 0
    progress.bar_style = "info"

    session = session_with_headers()
    downloaded = skipped = errors = 0

    status_html.value = f"<b>Status:</b> Running on {len(items)} rows..."
    log(f"Output folder: {out_dir}")

    for idx, row in enumerate(items, start=1):
        status_html.value = f"<b>Status:</b> Processing {idx}/{len(items)}"
        log(f"[{idx}/{len(items)}] row {row.get('rownum')}")

        result = process_row(session, row, out_dir, EMAIL_VALUE)
        APP_STATE["results"].append(result)

        if result["status"] == "downloaded":
            downloaded += 1
        elif result["status"] == "skipped":
            skipped += 1
        else:
            errors += 1

        log(f"   -> {result['status']}: {result['message']}")
        if result["saved_file"]:
            log(f"   -> saved: {result['saved_file']}")

        progress.value = idx
        show_results()
        time.sleep(delay)

    report_path = os.path.join(out_dir, "download_report.csv")
    write_report(APP_STATE["results"], report_path)

    zip_path = f"{out_dir}.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)
    zip_folder(out_dir, zip_path)

    APP_STATE["zip_path"] = zip_path
    APP_STATE["running"] = False
    zip_btn.disabled = False
    progress.bar_style = "success"

    status_html.value = (
        f"<b>Status:</b> Finished | Downloaded: {downloaded} | "
        f"Skipped: {skipped} | Errors: {errors}"
    )
    log("")
    log("Finished.")
    log(f"Downloaded: {downloaded}")
    log(f"Skipped: {skipped}")
    log(f"Errors: {errors}")
    log(f"Report: {report_path}")
    log(f"ZIP ready: {zip_path}")


def on_zip_clicked(b):
    zip_path = APP_STATE.get("zip_path")
    if not zip_path or not os.path.exists(zip_path):
        status_html.value = "<b>Status:</b> ZIP not available"
        log("ZIP not found.")
        return
    files.download(zip_path)


def on_clear_clicked(b):
    if APP_STATE["running"]:
        return
    APP_STATE["input_path"] = None
    APP_STATE["results"] = []
    APP_STATE["zip_path"] = None
    APP_STATE["output_dir"] = None
    file_html.value = "<b>Input file:</b> None"
    cols_html.value = "<b>Detected columns:</b> None"
    reset_outputs()
    show_results()


upload_btn.on_click(on_upload_clicked)
run_btn.on_click(on_run_clicked)
zip_btn.on_click(on_zip_clicked)
clear_btn.on_click(on_clear_clicked)

display(title_html)
display(widgets.HBox([email_widget, output_widget, delay_widget]))
display(widgets.HBox([upload_btn, run_btn, zip_btn, clear_btn]))
display(file_html)
display(cols_html)
display(status_html)
display(progress)
display(widgets.HTML("<b>Log</b>"))
display(log_output)
display(widgets.HTML("<b>Results</b>"))
display(table_output)

show_results()

HTML(value='<h2>Scholar PDF Downloader (Colab)</h2><p>Input can contain DOI, URL, full citation, or title.</p>…

HTML(value='<b>Input file:</b> None')

HTML(value='<b>Detected columns:</b> None')

HTML(value='<b>Status:</b> Ready')

IntProgress(value=0, description='Progress:', layout=Layout(width='700px'))

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saving Url.xlsx to Url (4).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>